# EGMS 2020–2024: pokaz dla Groningen i zachodniej NRW oraz droga do monitoringu co 6/12 dni

**Pytanie.** Co pokazuje European Ground Motion Service (EGMS, Sentinel-1 InSAR) w dwóch obszarach osiadania i wypiętrzania,
jak zgadza się z pomiarem naziemnym (GNSS, niwelacja) i po jakim czasie od nowych akwizycji osiadanie byłoby widoczne
ponad szumem, gdyby EGMS aktualizować co 6/12 dni, a nie raz w roku?

**Dane (otwarte).** EGMS Calibrated L2b (LOS) i Ortho L3 (pion), wydanie 2020–2024, Copernicus Land Monitoring Service ·
dobowe szeregi GNSS z Nevada Geodetic Laboratory · niwelacja Geobasis NRW · etap B: Sentinel-1 przetworzony w ASF HyP3 i MintPy.

**Zasady.** Parametry i progi są w `CHECK_CONFIG` (`step_05_colab_run.py`); sekcja niezatwierdzona przez autora zatrzymuje
zadania, które jej używają. Każde zadanie zapisuje się w rejestrze `gwl_runs` na Dysku; błąd zatrzymuje notatnik.
Pomiar naziemny potwierdza tempo osiadania widoczne w EGMS — EGMS jest produktem archiwalnym i nie wykrywa zdarzeń nagłych.

Uruchom: `Środowisko wykonawcze → Uruchom wszystko`. Pobranie EGMS i zlecenia HyP3 wymagają potwierdzenia (sekcje 2, 3, 5).

## 0. Środowisko
Montowanie Dysku, klon lub aktualizacja repozytorium na Dysku, świeży import modułów, pakiety etapu A.

In [1]:
import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
REPO_URL, BRANCH = 'https://github.com/geoworldlook/2_geoworldlook-egmscheck.git', 'claude/dreamy-dirac-jd4vue'
PROJECT_DIR = '/content/drive/MyDrive/GeoWorldLook/egmscheck'

def git(*args):
    # Błąd gita zatrzymuje notatnik (polecenie '!git' nie zatrzymuje: po nieudanym pull liczyłby stary kod).
    r = subprocess.run(['git', *args], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} -> kod {r.returncode}: {(r.stderr or r.stdout).strip()}")
    return r.stdout.strip()

if not os.path.exists(os.path.join(PROJECT_DIR, '.git')):
    os.makedirs(os.path.dirname(PROJECT_DIR), exist_ok=True)
    git('clone', '-q', '-b', BRANCH, REPO_URL, PROJECT_DIR)
else:
    git('-C', PROJECT_DIR, 'fetch', '-q', 'origin', BRANCH)
    git('-C', PROJECT_DIR, 'checkout', '-q', BRANCH)
    git('-C', PROJECT_DIR, 'pull', '-q', '--ff-only', 'origin', BRANCH)
print(git('-C', PROJECT_DIR, 'log', '-1', '--format=Kod: %h %ci %s'))
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
for m in [m for m in sys.modules if m.startswith('step_0')]:
    del sys.modules[m]                      # po git pull importujemy świeży kod
import copy, json
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display, Image, Markdown
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)
from step_05_colab_run import (setup_runtime, ensure_packages, run_task, run_history, registry_summary, region_paths,
                               hyp3_planned_tracks, STAGE_A_PACKAGES, STAGE_B_PACKAGES,
                               task_select_gnss, task_gnss_download, task_levelling, task_egms_search, task_egms_download,
                               task_extract, task_checks, task_compare_gnss, task_compare_levelling,
                               task_monitoring_archive, task_figures, task_hyp3_plan, task_hyp3_submit,
                               task_hyp3_download, task_mintpy, task_hyp3_compare, task_summary)
rt = setup_runtime(PROJECT_DIR, STAGE_A_PACKAGES)
print('Projekt:', rt['PROJECT_DIR'], '| commit:', rt['GIT_COMMIT'])

Mounted at /content/drive
Kod: a7b13b9 2026-10-12 09:38:31 +0200 Utworzono za pomocą Colab. Zapis wyników przebiegu testu.
Projekt: /content/drive/MyDrive/GeoWorldLook/egmscheck | commit: a7b13b9+dirty


## 1. Konfiguracja i zatwierdzenia
`cfg` to kopia `CHECK_CONFIG`. Tabela pokazuje, które sekcje autor zatwierdził (data) i jakie decyzje są otwarte
(wartości `None`). Wartości oznaczone w kodzie `[p]` to propozycje; zmienia się je w `step_05_colab_run.py` z datą zatwierdzenia,
nigdy po obejrzeniu wyników.

In [2]:
from step_05_colab_run import CHECK_CONFIG, approvals_table
cfg = copy.deepcopy(CHECK_CONFIG)
display(approvals_table(cfg))

,section,approved,status,open_decisions
0,egms,2026-10-11,zatwierdzone,-
1,regions.groningen,2026-10-10,zatwierdzone,-
2,regions.nrw_west,2026-10-12,zatwierdzone,-
3,gnss,2026-10-12,zatwierdzone,-
4,method,2026-10-12,zatwierdzone,-
5,thresholds,2026-10-12,zatwierdzone,-
6,monitoring,2026-10-12,zatwierdzone,-
7,hyp3,2026-10-12,zatwierdzone,-


## 2. Groningen (wydobycie gazu)
Kolejność: wyszukanie plików EGMS L2b i L3 (bez pobierania) z planem miejsca na Dysku → potwierdzenie rozmiaru →
pobranie → stacje GNSS według reguły z konfiguracji → dobowe szeregi GNSS →
ponowne wyszukanie i pobranie plików dla punktów spoza obszaru → wycinki i kontrole. Pobranie EGMS wymaga tylko
zatwierdzonego wydania i obszaru, więc może ruszyć przed decyzjami o GNSS i progach.
Pobranie wymaga wpisania liczby MB z planu w komórce poniżej (ochrona przed przypadkowym pobraniem wielu GB).

In [3]:
P_GRONINGEN = region_paths(rt, cfg, 'groningen')
run_task(rt, 'egms_search_groningen', task_egms_search, rt, cfg, 'groningen')
if P_GRONINGEN['plan'].exists():
    plan = pd.read_csv(P_GRONINGEN['plan'])
    display(plan.groupby(['query_key', 'status']).agg(files=('filename', 'size'), MB=('filesize', lambda s: round(s.sum() / 1e6, 1))))

             files      MB
query_key                 
L2B             46  5305.9
L3:ORTHO-UP      2   110.1
[OK] egms_search_groningen (7.4 s) 48 plików, łącznie 5416.0 MB, do pobrania 0.0 MB ({'existing': 48}); po akceptacji ustaw CONFIRM_MB_GRONINGEN = 0


,,files,MB
query_key,status,,
L2B,existing,46,5305.9
L3:ORTHO-UP,existing,2,110.1


### Potwierdzenie rozmiaru pobrania: Groningen
Wpisz liczbę MB z komunikatu `egms_search_groningen` (suma plików brakujących, zaokrąglona). Inna liczba albo `None`
zatrzymuje notatnik z komunikatem, jaką wartość wpisać. Gdy po wyborze stacji ponowne wyszukanie znajdzie pliki dla
punktów spoza obszaru, notatnik zatrzyma się z nową liczbą MB do wpisania tutaj; pobrane pliki zostają na Dysku.
Bez tokenu CLMS notatnik czyta istniejące wycinki z `data/03_Extracts/`.

In [5]:
CONFIRM_MB_GRONINGEN = None   # np. 4321 - liczba MB z planu

In [6]:
run_task(rt, 'egms_download_groningen', task_egms_download, rt, cfg, 'groningen', CONFIRM_MB_GRONINGEN)
run_task(rt, 'select_gnss_groningen', task_select_gnss, rt, cfg, 'groningen')
run_task(rt, 'gnss_download_groningen', task_gnss_download, rt, cfg, 'groningen')
display(pd.read_csv(P_GRONINGEN['sites'], dtype={'site': str}).head(30))
# Punkty spoza obszaru (np. stacje stabilne z okolicy): ponowne wyszukanie dobiera ich pliki. Gdy potrzeba nowych plików,
# pobranie zatrzymuje się z liczbą MB do wpisania w CONFIRM_MB_GRONINGEN; po wpisaniu uruchom notatnik ponownie (pobrane pliki zostają).
run_task(rt, 'egms_search_sites_groningen', task_egms_search, rt, cfg, 'groningen')
run_task(rt, 'egms_download_sites_groningen', task_egms_download, rt, cfg, 'groningen', CONFIRM_MB_GRONINGEN)
run_task(rt, 'extract_groningen', task_extract, rt, cfg, 'groningen')
run_task(rt, 'checks_groningen', task_checks, rt, cfg, 'groningen')
display(pd.read_csv(P_GRONINGEN['checks']))

[SKIPPED] egms_download_groningen (0.7 s) wszystkie 48 pliki z planu są na Dysku i w manifeście
MIDAS: migawka z Dysku midas_20261012.IGS.txt (dostępne: ['midas_20261012.IGS.txt']); bez ponownego pobrania - nowszy stan MIDAS zmienia wybór stacji (decyzja autora)
Stacje stabilne spełniające regułę w regionie kontrolnym: 30; wybrano 8 najbliższych stacji osiadającej.
[OK] select_gnss_groningen (5.1 s) 35 stacji GNSS ({'osiadająca': 27, 'stabilna': 8}), reguła subsiding_and_stable; gwl_sites: +0/~0
site      first       last  days
AME1 2006-05-10 2024-12-17  6534
VEEN 2007-11-14 2026-09-26  6745
BRNH 2018-04-24 2026-06-27  2949
STED 2014-03-31 2026-06-27  4321
ZEER 2014-03-30 2026-06-27  4416
DZYL 2005-09-21 2026-09-12  7479
LEER 2018-04-18 2026-06-27  2967
FROO 2014-02-28 2026-06-27  4484
MODD 2007-01-01 2026-06-27  7032
AME2 2014-09-12 2026-06-27  4129
BIER 2018-04-18 2026-06-27  2944
ANJM 2007-01-01 2026-06-27  7066
UITH 2018-04-19 2026-06-27  2968
DZY1 2014-03-19 2026-06-27  4472
OLDO

,site,site_type,group,lat_deg,lon_deg,x_3035,y_3035,source,vu_mm_yr,svu_mm_yr,t_first,t_last,n_steps,dist_km
0,AME1,gnss,osiadająca,53.464433,5.921342,4.050254e+06,3.380573e+06,GNSS_NGL,-6.849,0.474,2006.3546,2024.6461,0.0,0.000000
1,VEEN,gnss,osiadająca,53.104277,6.865016,4.111095e+06,3.337410e+06,GNSS_NGL,-6.407,0.543,2007.8686,2024.6461,0.0,0.000000
2,BRNH,gnss,osiadająca,53.310245,6.677875,4.099633e+06,3.360867e+06,GNSS_NGL,-5.944,0.920,2018.3107,2024.6461,0.0,0.000000
3,STED,gnss,osiadająca,53.334332,6.701538,4.101332e+06,3.363473e+06,GNSS_NGL,-5.379,0.700,2014.2450,2024.6461,0.0,0.000000
4,ZEER,gnss,osiadająca,53.346304,6.737996,4.103819e+06,3.364695e+06,GNSS_NGL,-5.277,0.721,2014.2423,2024.6461,0.0,0.000000
5,DZYL,gnss,osiadająca,53.320040,6.940434,4.117163e+06,3.361187e+06,GNSS_NGL,-4.811,0.499,2005.7221,2024.6461,2.0,0.000000
6,LEER,gnss,osiadająca,53.351685,6.817261,4.109119e+06,3.365058e+06,GNSS_NGL,-4.578,0.920,2018.2943,2024.6461,0.0,0.000000
7,FROO,gnss,osiadająca,53.187723,6.781618,4.105930e+06,3.346933e+06,GNSS_NGL,-4.202,0.672,2014.1602,2024.6461,0.0,0.000000
8,MODD,gnss,osiadająca,53.405358,6.067502,4.059584e+06,3.373471e+06,GNSS_NGL,-4.159,0.487,2007.0007,2024.6461,0.0,0.000000
9,AME2,gnss,osiadająca,53.483304,5.866851,4.046762e+06,3.382875e+06,GNSS_NGL,-3.972,0.681,2014.6968,2024.6461,0.0,0.000000


             files      MB
query_key                 
L2B             46  5305.9
L3:ORTHO-UP      2   110.1
[OK] egms_search_sites_groningen (2.4 s) 48 plików, łącznie 5416.0 MB, do pobrania 0.0 MB ({'existing': 48}); po akceptacji ustaw CONFIRM_MB_GRONINGEN = 0
[SKIPPED] egms_download_sites_groningen (0.2 s) wszystkie 48 pliki z planu są na Dysku i w manifeście
EGMS_L2b_015_0317_IW1_VV_2020_2024_1.zip: 1035 punktów w promieniu 200 m
EGMS_L2b_037_0757_IW2_VV_2020_2024_1.zip: 1854 punktów w promieniu 200 m
EGMS_L2b_015_0317_IW2_VV_2020_2024_1.zip: 121 punktów w promieniu 200 m
EGMS_L2b_110_0756_IW1_VV_2020_2024_1.zip: 334 punktów w promieniu 200 m
EGMS_L2b_015_0312_IW2_VV_2020_2024_1.zip: 0 punktów w promieniu 200 m
EGMS_L2b_037_0760_IW2_VV_2020_2024_1.zip: 0 punktów w promieniu 200 m
EGMS_L2b_037_0754_IW1_VV_2020_2024_1.zip: 0 punktów w promieniu 200 m
EGMS_L2b_139_0759_IW3_VV_2020_2024_1.zip: 894 punktów w promieniu 200 m
EGMS_L2b_037_0759_IW1_VV_2020_2024_1.zip: 0 punktów w promieniu

,check,track,value,threshold,pass
0,A1 |norma LOS - 1|,15,0.0006442075016526161,0.01,True
1,A2 |los_up - cos(theta)|,15,0.0005320116040982148,0.01,True
2,A1 |norma LOS - 1|,37,0.0007387271411054552,0.01,True
3,A2 |los_up - cos(theta)|,37,0.0005245945152601017,0.01,True
4,A1 |norma LOS - 1|,88,0.0006198079190717909,0.01,True
5,A2 |los_up - cos(theta)|,88,0.0005160677398626801,0.01,True
6,A1 |norma LOS - 1|,110,0.0005123687391375142,0.01,True
7,A2 |los_up - cos(theta)|,110,0.0005246013573054764,0.01,True
8,A1 |norma LOS - 1|,139,0.0006557149810682006,0.01,True
9,A2 |los_up - cos(theta)|,139,0.0005511028193481149,0.01,True


### Porównanie z GNSS
Każda para stacja × ścieżka L2b (LOS, bufory 100 i 200 m) i stacja × komórka L3 (pion). Para bez punktów EGMS ma wiersz
ze statusem — nie znika z tabeli. Werdykt L2b według kryteriów K1–K4 (jak E1); dla L3 zdefiniowany jest tylko próg K1.

In [7]:
run_task(rt, 'compare_gnss_groningen', task_compare_gnss, rt, cfg, 'groningen')
if P_GRONINGEN['gnss_compare'].exists():
    res = pd.read_csv(P_GRONINGEN['gnss_compare'], dtype={'site': str})
    cols = ['product', 'site', 'group', 'track', 'radius_m', 'n_points', 'v_gnss_los', 'v_egms', 'dv', 'series_diff_std', 'meets_A6', 'status']
    display(res[[c for c in cols if c in res]].round(2))
    for key in ('verdict_l2b', 'verdict_l3u'):
        if P_GRONINGEN[key].exists():
            display(pd.read_csv(P_GRONINGEN[key]))
            print('WNIOSEK:', P_GRONINGEN[key].with_suffix('.txt').read_text())

                                           kryterium                                 wynik          próg spełnione  n_par
                         K1 zgodność tempa osiadania          99% par z |dv| <= 1.5 mm/rok        >= 80%      True     93
                      K2 zgodność szeregów czasowych            mediana std różnicy 4.7 mm     <= 5.0 mm      True     93
                   Bias i rozrzut dv (informacyjnie) mediana -0.00, MAD*1.4826 0.81 mm/rok             -      None     93
       K3 kontrola negatywna (stacje stabilne, |dv|)                  max |dv| 1.62 mm/rok <= 1.5 mm/rok     False     32
K4 nachylenie v_EGMS względem v_GNSS (informacyjnie)              0.94 (wyraz wolny -0.14)     bliskie 1      None    125
                                           kryterium                                 wynik                                                                próg spełnione  n_par
                         K1 zgodność tempa osiadania          96% par z |dv| <= 1.5 mm/rok  

,product,site,group,track,radius_m,n_points,v_gnss_los,v_egms,dv,series_diff_std,meets_A6,status
0,EGMS_L2b_2020_2024,AME1,osiadająca,15,100.00,64,-5.27,-5.02,-0.24,4.69,True,ok
1,EGMS_L2b_2020_2024,AME1,osiadająca,15,200.00,68,-5.27,-5.00,-0.27,4.68,True,ok
2,EGMS_L2b_2020_2024,AME1,osiadająca,37,100.00,131,-4.32,-4.73,0.41,4.93,True,ok
3,EGMS_L2b_2020_2024,AME1,osiadająca,37,200.00,152,-4.32,-4.75,0.43,4.91,True,ok
4,EGMS_L2b_2020_2024,AME1,osiadająca,88,100.00,97,-4.88,-4.25,-0.63,4.34,True,ok
...,...,...,...,...,...,...,...,...,...,...,...,...
380,EGMS_L3U_2020_2024,TGEM,stabilna,L3,70.71,1,-1.09,-1.79,0.70,5.99,True,ok
381,EGMS_L3U_2020_2024,TGBF,stabilna,L3,70.71,1,-0.86,-2.53,1.67,6.07,True,ok
382,EGMS_L3U_2020_2024,AMEL,stabilna,L3,70.71,1,-0.82,-1.01,0.19,6.12,True,ok
383,EGMS_L3U_2020_2024,RDN1,stabilna,L3,70.71,1,-0.14,-0.70,0.56,5.77,True,ok


,kryterium,wynik,próg,spełnione,n_par
0,K1 zgodność tempa osiadania,99% par z |dv| <= 1.5 mm/rok,>= 80%,True,93
1,K2 zgodność szeregów czasowych,mediana std różnicy 4.7 mm,<= 5.0 mm,True,93
2,Bias i rozrzut dv (informacyjnie),"mediana -0.00, MAD*1.4826 0.81 mm/rok",-,NaN,93
3,"K3 kontrola negatywna (stacje stabilne, |dv|)",max |dv| 1.62 mm/rok,<= 1.5 mm/rok,False,32
4,K4 nachylenie v_EGMS względem v_GNSS (informac...,0.94 (wyraz wolny -0.14),bliskie 1,NaN,125


WNIOSEK: CZĘŚCIOWO: tempo osiadania zgodne, ale szeregi czasowe lub kontrola negatywna poza progiem.



,kryterium,wynik,próg,spełnione,n_par
0,K1 zgodność tempa osiadania,96% par z |dv| <= 1.5 mm/rok,>= 80%,True,24
1,K2 zgodność szeregów czasowych,mediana std różnicy 5.6 mm,brak progu w CHECK_CONFIG['thresholds']['gnss_...,NaN,24
2,Bias i rozrzut dv (informacyjnie),"mediana -0.21, MAD*1.4826 0.54 mm/rok",-,NaN,24
3,K3 kontrola negatywna (stacje stabilne),max |v_EGMS| 2.53 mm/rok,brak progu w CHECK_CONFIG['thresholds']['gnss_...,NaN,8
4,K4 nachylenie v_EGMS względem v_GNSS (informac...,0.96 (wyraz wolny -0.20),bliskie 1,NaN,32


WNIOSEK: L3 (pion): K1 spełnione - tempo osiadania z EGMS L3 zgodne z GNSS (U) w granicach progu.



## 3. Zachodnia NRW (odkrywki węgla brunatnego, wypiętrzanie)
Kolejność: wyszukanie plików EGMS L2b i L3 (bez pobierania) z planem miejsca na Dysku → potwierdzenie rozmiaru →
pobranie → stacje GNSS według reguły z konfiguracji → dobowe szeregi GNSS → niwelacja (repery w obszarze) →
ponowne wyszukanie i pobranie plików dla punktów spoza obszaru → wycinki i kontrole. Pobranie EGMS wymaga tylko
zatwierdzonego wydania i obszaru, więc może ruszyć przed decyzjami o GNSS, niwelacji i progach.
Pobranie wymaga wpisania liczby MB z planu w komórce poniżej (ochrona przed przypadkowym pobraniem wielu GB).

In [10]:
P_NRW_WEST = region_paths(rt, cfg, 'nrw_west')
run_task(rt, 'egms_search_nrw_west', task_egms_search, rt, cfg, 'nrw_west')
if P_NRW_WEST['plan'].exists():
    plan = pd.read_csv(P_NRW_WEST['plan'])
    display(plan.groupby(['query_key', 'status']).agg(files=('filename', 'size'), MB=('filesize', lambda s: round(s.sum() / 1e6, 1))))

Brak /content/drive/MyDrive/GeoWorldLook/egmscheck/data/03_Extracts/nrw_west/sites.csv: wyszukanie EGMS tylko dla obszaru regionu. Po wyborze stacji (task_select_gnss) uruchom ponownie egms_search, aby dobrać pliki dla punktów spoza obszaru.
             files       MB
query_key                  
L2B             36  17325.3
L3:ORTHO-UP      4    376.9
[OK] egms_search_nrw_west (2.0 s) 40 plików, łącznie 17702.2 MB, do pobrania 17702.2 MB ({'missing': 40}); po akceptacji ustaw CONFIRM_MB_NRW_WEST = 17702


,,files,MB
query_key,status,,
L2B,missing,36,17325.3
L3:ORTHO-UP,missing,4,376.9


### Potwierdzenie rozmiaru pobrania: Zachodnia NRW
Wpisz liczbę MB z komunikatu `egms_search_nrw_west` (suma plików brakujących, zaokrąglona). Inna liczba albo `None`
zatrzymuje notatnik z komunikatem, jaką wartość wpisać. Gdy po wyborze stacji ponowne wyszukanie znajdzie pliki dla
punktów spoza obszaru, notatnik zatrzyma się z nową liczbą MB do wpisania tutaj; pobrane pliki zostają na Dysku.
Bez tokenu CLMS notatnik czyta istniejące wycinki z `data/03_Extracts/`.

In [11]:
CONFIRM_MB_NRW_WEST = 17702   # np. 4321 - liczba MB z planu

In [12]:
run_task(rt, 'egms_download_nrw_west', task_egms_download, rt, cfg, 'nrw_west', CONFIRM_MB_NRW_WEST)
run_task(rt, 'select_gnss_nrw_west', task_select_gnss, rt, cfg, 'nrw_west')
run_task(rt, 'gnss_download_nrw_west', task_gnss_download, rt, cfg, 'nrw_west')
run_task(rt, 'levelling_nrw_west', task_levelling, rt, cfg, 'nrw_west')
display(pd.read_csv(P_NRW_WEST['sites'], dtype={'site': str}).head(30))
# Punkty spoza obszaru (np. stacje stabilne z okolicy): ponowne wyszukanie dobiera ich pliki. Gdy potrzeba nowych plików,
# pobranie zatrzymuje się z liczbą MB do wpisania w CONFIRM_MB_NRW_WEST; po wpisaniu uruchom notatnik ponownie (pobrane pliki zostają).
run_task(rt, 'egms_search_sites_nrw_west', task_egms_search, rt, cfg, 'nrw_west')
run_task(rt, 'egms_download_sites_nrw_west', task_egms_download, rt, cfg, 'nrw_west', CONFIRM_MB_NRW_WEST)
run_task(rt, 'extract_nrw_west', task_extract, rt, cfg, 'nrw_west')
run_task(rt, 'checks_nrw_west', task_checks, rt, cfg, 'nrw_west')
display(pd.read_csv(P_NRW_WEST['checks']))

Brak /content/drive/MyDrive/GeoWorldLook/egmscheck/data/03_Extracts/nrw_west/sites.csv: wyszukanie EGMS tylko dla obszaru regionu. Po wyborze stacji (task_select_gnss) uruchom ponownie egms_search, aby dobrać pliki dla punktów spoza obszaru.
[OK] egms_download_nrw_west (1167.0 s) potwierdzone 17702 MB; {'downloaded': 40}
MIDAS: migawka z Dysku midas_20261012.IGS.txt (dostępne: ['midas_20261012.IGS.txt']); bez ponownego pobrania - nowszy stan MIDAS zmienia wybór stacji (decyzja autora)
Stacje GNSS w obszarze (5.9, 50.7, 7.0, 51.3): 17; od <= 2020.0: 12; do >= 2024.25: 11; svu <= 1.0 mm/rok: 10; skoki <= 2: 16; wszystkie warunki: 6
Grupy stacji GNSS: {'inna': 5, 'stabilna': 1}
[OK] select_gnss_nrw_west (1.9 s) 6 stacji GNSS ({'inna': 5, 'stabilna': 1}), reguła all_quality; gwl_sites: +6/~0
site      first       last  days
SEL2 2013-02-02 2026-09-19  4607
AACH 2002-11-20 2026-09-19  7721
TIT2 2014-12-11 2026-09-19  4199
VIER 2002-11-20 2026-09-19  7647
KERK 2006-10-18 2026-09-12  7142
ROE

ERROR:EGMSCheck_Run:Zadanie levelling_nrw_west zakończone błędem: ValueError: /content/drive/MyDrive/GeoWorldLook/egmscheck/data/02_Raw_Ground/4_levelling_nrw/hfp_pl.csv (Punktliste): błędny CSV w linii 2944: ';' expected after '"'


[ERROR] levelling_nrw_west (1.1 s) ValueError: /content/drive/MyDrive/GeoWorldLook/egmscheck/data/02_Raw_Ground/4_levelling_nrw/hfp_pl.csv (Punktliste): błędny CSV w linii 2944: ';' expected after '"'


ValueError: /content/drive/MyDrive/GeoWorldLook/egmscheck/data/02_Raw_Ground/4_levelling_nrw/hfp_pl.csv (Punktliste): błędny CSV w linii 2944: ';' expected after '"'

### Porównanie z GNSS
Każda para stacja × ścieżka L2b (LOS, bufory 100 i 200 m) i stacja × komórka L3 (pion). Para bez punktów EGMS ma wiersz
ze statusem — nie znika z tabeli. Werdykt L2b według kryteriów K1–K4 (jak E1); dla L3 zdefiniowany jest tylko próg K1.

In [ ]:
run_task(rt, 'compare_gnss_nrw_west', task_compare_gnss, rt, cfg, 'nrw_west')
if P_NRW_WEST['gnss_compare'].exists():
    res = pd.read_csv(P_NRW_WEST['gnss_compare'], dtype={'site': str})
    cols = ['product', 'site', 'group', 'track', 'radius_m', 'n_points', 'v_gnss_los', 'v_egms', 'dv', 'series_diff_std', 'meets_A6', 'status']
    display(res[[c for c in cols if c in res]].round(2))
    for key in ('verdict_l2b', 'verdict_l3u'):
        if P_NRW_WEST[key].exists():
            display(pd.read_csv(P_NRW_WEST[key]))
            print('WNIOSEK:', P_NRW_WEST[key].with_suffix('.txt').read_text())

### Porównanie z niwelacją
Δh niwelacji i EGMS L3 (pion) między pierwszą i ostatnią epoką reperu w oknie dat EGMS; kryteria N1–N4
(mediana |różnicy|, zgodność znaku, kontrola negatywna na reperach stabilnych, bias prędkości).

In [ ]:
run_task(rt, 'compare_levelling_nrw_west', task_compare_levelling, rt, cfg, 'nrw_west')
display(pd.read_csv(P_NRW_WEST['levelling_summary']))

## 4. Monitoring na danych archiwalnych i ryciny
Wspólny format szeregów (`timeseries_long.csv.gz`): EGMS L2b (mediana punktów w buforze), EGMS L3 (komórka), GNSS w LOS
i w pionie, niwelacja. Próbkowanie akwizycji (6 dni do końca Sentinel-1B, potem 12 dni) i „ile czasu do wykrycia”:
prędkość z rosnącego okna z niepewnością poprawioną na autokorelację (AR(1)), dla EGMS i GNSS w tych samych datach.

In [ ]:
for region in ('groningen', 'nrw_west'):
    run_task(rt, f'monitoring_archive_{region}', task_monitoring_archive, rt, cfg, region)
    run_task(rt, f'figures_{region}', task_figures, rt, cfg, region)
    P = region_paths(rt, cfg, region)
    display(pd.read_csv(P['ttd'], dtype={'site': str, 'track': str}).pivot_table(index=['site', 'track'], columns=['source', 'start'], values='ttd_months', aggfunc='first'))
    for png in sorted(P['figures'].glob(f'{region}_*.png'))[:12]:
        print(png.name); display(Image(filename=str(png), width=800))

## 5. Etap B: własne przetworzenie Sentinel-1 (ASF HyP3 + MintPy)
Ścieżki z planu EGMS L2b regionu z `cfg['hyp3']['region']`. Bursty pokrywające punkty naziemne, sieć par SBAS i zlecenia
HyP3 (płatne kredytami): najpierw plan bez wysyłania, potem potwierdzenie liczby zleceń na ścieżkę. Zlecenia przetwarzają się
godzinami — pobranie i MintPy uruchom ponownie, gdy zlecenia mają status SUCCEEDED. Zakładka z EGMS (2024) służy
do sprawdzenia i dowiązania szeregu; dalsze daty to „monitoring”, którego EGMS jeszcze nie ma.

In [ ]:
ensure_packages(STAGE_B_PACKAGES)
HYP3_REGION = cfg['hyp3']['region']
run_task(rt, f'hyp3_plan_{HYP3_REGION}', task_hyp3_plan, rt, cfg, HYP3_REGION)
display(pd.read_csv(region_paths(rt, cfg, HYP3_REGION)['hyp3_plan']))

### Potwierdzenie zleceń HyP3
Wpisz słownik `{ścieżka: liczba zleceń}` z komunikatu `hyp3_plan_...`. `None` albo inna liczba zatrzymuje notatnik.

In [ ]:
CONFIRM_JOBS = None   # np. {'37': 54, '88': 54}

In [ ]:
for track in hyp3_planned_tracks(rt, cfg, HYP3_REGION):
    run_task(rt, f'hyp3_submit_{HYP3_REGION}_{track}', task_hyp3_submit, rt, cfg, HYP3_REGION, track,
             None if CONFIRM_JOBS is None else CONFIRM_JOBS.get(track))
for track in hyp3_planned_tracks(rt, cfg, HYP3_REGION):
    run_task(rt, f'hyp3_download_{HYP3_REGION}_{track}', task_hyp3_download, rt, cfg, HYP3_REGION, track)
    run_task(rt, f'mintpy_{HYP3_REGION}_{track}', task_mintpy, rt, cfg, HYP3_REGION, track)
    run_task(rt, f'hyp3_compare_{HYP3_REGION}_{track}', task_hyp3_compare, rt, cfg, HYP3_REGION, track)
P = region_paths(rt, cfg, HYP3_REGION)
for f in sorted(P['outputs'].glob('hyp3_align_*.csv')):
    print(f.name); display(pd.read_csv(f).round(2))
for png in sorted(P['figures'].glob(f'{HYP3_REGION}_hyp3_series_*.png'))[:6]:
    print(png.name); display(Image(filename=str(png), width=800))

## 6. Podsumowanie uruchomienia
`run_summary.md` i `run_summary.json` (wersje bibliotek, konfiguracja, wydanie EGMS, commit, zatwierdzenia, werdykty,
ostatnie uruchomienia) w `data/05_Final_Outputs/egmscheck/`.

In [ ]:
run_task(rt, 'summary', task_summary, rt, cfg)
display(run_history(rt, 30))
display(registry_summary(rt))
display(Markdown(Path(PROJECT_DIR, cfg['project']['outputs_dir'], 'run_summary.md').read_text()[:6000]))

## Jak czytać wynik
- Zgodność EGMS z GNSS i niwelacją dotyczy powolnego osiadania i wypiętrzania tam, gdzie są punkty pomiarowe; EGMS L2b/L3
  jest kalibrowany modelem GNSS, więc zgodność prędkości bezwzględnej nie jest w pełni niezależna.
- „Czas do wykrycia” mówi, po ilu miesiącach nowych akwizycji trend przekracza k × niepewność; nie dotyczy zdarzeń nagłych.
- Etap B pokazuje, że ten sam szereg można przedłużać poza wydanie EGMS (monitoring), po sprawdzeniu na zakładce z EGMS.